In [120]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

import time
import math
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from transformers import AutoTokenizer, AutoModel

In [102]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SOS_TOKEN = 0
EOS_TOKEN = 1

MAX_LENGTH = 10

eng_prefixes = (
    "i am ",
    "i m ",
    "he is",
    "he s ",
    "she is",
    "she s ",
    "you are",
    "you re ",
    "we are",
    "we re ",
    "they are",
    "they re ",
)

In [103]:
class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2

    def add_sentence(self, sentence):
        for word in sentence.split(" "):
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [104]:
def unicode_to_ascii(s):
    return "".join(
        c for c in unicodedata.normalize("NFD", s) if unicodedata.category(c) != "Mn"
    )


def normalise_string(s):
    s = unicode_to_ascii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

In [105]:
def read_langs(lang1, lang2, reverse=False):
    print("Reading lines...")

    lines = (
        open(f"data/{lang1}-{lang2}.txt", encoding="utf-8").read().strip().split("\n")
    )

    pairs = [[normalise_string(s) for s in l.split("\t")] for l in lines]

    if reverse:
        pairs = [list(reversed(p)) for p in pairs]
        input_lang = Lang(lang2)
        output_lang = Lang(lang1)
    else:
        input_lang = Lang(lang1)
        output_lang = Lang(lang2)

    return input_lang, output_lang, pairs

In [106]:
def filter_pair(p):
    return (
        len(p[0].split(" ")) < MAX_LENGTH
        and len(p[1].split(" ")) < MAX_LENGTH
        and p[1].startswith(eng_prefixes)
    )


def filter_pairs(pairs):
    return [pair for pair in pairs if filter_pair(pair)]

In [107]:
def prepare_data(lang1, lang2, reverse=False):
    input_lang, output_lang, pairs = read_langs(lang1, lang2, reverse)
    print(f"Read {len(pairs)} sentence pairs")
    pairs = filter_pairs(pairs)
    print(f"Trimmed to {len(pairs)} sentence pairs")
    print("Counting words...")
    for pair in pairs:
        input_lang.add_sentence(pair[0])
        output_lang.add_sentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [108]:
class Encoder(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(Encoder, self).__init__()
        self.embed = nn.Embedding(input_size, hidden_size)
        self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embed(input))
        output, hidden = self.gru(embedded)
        return output, hidden

In [109]:
class DotProductAttention(nn.Module):
    def __init__(self):
        super(DotProductAttention, self).__init__()

    def forward(self, encoder_outputs, hidden):

        hidden = hidden.permute(1,0,2) # [batch, 1 ,hidden_size]

        # [batch, 1, hidden] x [batch, hidden, seq_length] = [batch, 1, seq_length]
        attention_scores = torch.bmm(hidden, encoder_outputs.transpose(1,2))
        attention_weights = F.softmax(attention_scores, dim=2)

        # [batch, 1, seq_length] x [batch, seq_length, hidden] = [batch, 1, hidden_size]
        attention_vector = torch.bmm(attention_weights, encoder_outputs)

        return attention_vector, attention_weights


In [110]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, hidden_size):
        super(ScaledDotProductAttention, self).__init__()

        self.W_q = nn.Linear(hidden_size, hidden_size)
        self.W_k = nn.Linear(hidden_size, hidden_size)
        self.W_v = nn.Linear(hidden_size, hidden_size)

        self.scale = hidden_size ** 0.5

    def forward(self, encoder_outputs, hidden):

        hidden = hidden.permute(1,0,2) # [batch, 1 ,hidden_size]

        Q = self.W_q(hidden)
        K = self.W_k(encoder_outputs)
        V = self.W_v(encoder_outputs)

        # [batch, 1, hidden] x [batch, hidden, seq_length] = [batch, 1, seq_length]
        attention_scores = torch.bmm(Q, K.transpose(1,2)) / self.scale
        attention_weights = F.softmax(attention_scores, dim=2)

        # [batch, 1, seq_length] x [batch, seq_length, hidden] = [batch, 1, hidden_size]
        attention_vector = torch.bmm(attention_weights, V)

        return attention_vector, attention_weights


In [111]:
class Decoder(nn.Module):
    def __init__(self, hidden_size, output_size, attention=None):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = attention
        if self.attention is not None:
            self.gru = nn.GRU(2*hidden_size, hidden_size, batch_first=True)
        else:
            self.gru = nn.GRU(hidden_size, hidden_size, batch_first=True)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_TOKEN)
        decoder_hidden = encoder_hidden
        decoder_outputs = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden  = self.forward_step(decoder_input, encoder_outputs, decoder_hidden)
            decoder_outputs.append(decoder_output)

            if target_tensor is not None:
                # Teacher forcing: Feed the target as the next input
                decoder_input = target_tensor[:, i].unsqueeze(1) # Teacher forcing
            else:
                # Without teacher forcing: use its own predictions as the next input
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()  # detach from history as input

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        return decoder_outputs, decoder_hidden, None # We return `None` for consistency in the training loop

    def forward_step(self, input, encoder_outputs, hidden):
        embedded = self.embedding(input)
        embedded = F.relu(embedded)
        if self.attention is not None:
            attention_vector, _ = self.attention(encoder_outputs, hidden)
            rnn_input = torch.cat([embedded, attention_vector], dim=2)
            output, hidden = self.gru(rnn_input, hidden)
        else:
            output, hidden = self.gru(embedded, hidden)
        output = self.out(output)
        return output, hidden

In [112]:
def indexes_from_sentences(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(" ")]


def tensor_from_sentence(lang, sentence):
    indexes = indexes_from_sentences(lang, sentence)
    indexes.append(EOS_TOKEN)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)


def tensors_from_pair(input_lang, output_lang, pair):
    input_tensor = tensor_from_sentence(input_lang, pair[0])
    target_tensor = tensor_from_sentence(output_lang, pair[1])
    return (input_tensor, target_tensor)

In [113]:
def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepare_data("eng", "fra", True)

    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexes_from_sentences(input_lang, inp)
        tgt_ids = indexes_from_sentences(output_lang, tgt)
        inp_ids.append(EOS_TOKEN)
        tgt_ids.append(EOS_TOKEN)
        input_ids[idx, : len(inp_ids)] = inp_ids
        target_ids[idx, : len(tgt_ids)] = tgt_ids
    train_data = TensorDataset(
        torch.LongTensor(input_ids).to(device), torch.LongTensor(target_ids).to(device)
    )
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(
        train_data, sampler=train_sampler, batch_size=batch_size
    )
    return input_lang, output_lang, pairs, train_dataloader

In [114]:
def as_minutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return "%dm %ds" % (m, s)


def time_since(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return "%s (- %s)" % (as_minutes(s), as_minutes(rs))


def show_plot(points):
    plt.figure()
    fig, ax = plt.subplots()
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)
    # plt.savefig("loss_plot.png")
    plt.close()


In [115]:
def train_epoch(
    dataloader, encoder, decoder, encoder_optimizer, decoder_optimizer, criterion
):
    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data
        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)), target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [116]:
def train(
    train_dataloader,
    encoder,
    decoder,
    n_epochs,
    learning_rate=0.001,
    print_every=100,
    plot_every=100,
):
    start = time.time()
    plot_losses = []
    print_loss_total = 0
    plot_loss_total = 0

    encoder_optimzer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimzer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(
            train_dataloader,
            encoder,
            decoder,
            encoder_optimzer,
            decoder_optimzer,
            criterion,
        )
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print(
                "%s (%d %d%%) %.4f"
                % (
                    time_since(start, epoch / n_epochs),
                    epoch,
                    epoch / n_epochs * 100,
                    print_loss_avg,
                )
            )

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    show_plot(plot_losses)

In [117]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensor_from_sentence(input_lang, sentence)

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(
            encoder_outputs, encoder_hidden
        )

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_TOKEN:
                decoded_words.append("<EOS>")
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn

In [118]:
def evaluate_randomly(encoder, decoder, input_lang, output_lang, pairs, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print(">", pair[0])
        print("=", pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = " ".join(output_words)
        print("<", output_sentence)
        print("")

In [119]:
hidden_size = 64
batch_size = 256
n_epochs = 80

input_lang, output_lang, pairs, train_dataloader = get_dataloader(batch_size)

encoder = Encoder(input_lang.n_words, hidden_size).to(device)
attention = ScaledDotProductAttention(hidden_size)
decoder = Decoder(hidden_size, output_lang.n_words, attention).to(device)

train(train_dataloader, encoder, decoder, n_epochs, print_every=5, plot_every=5)
encoder.eval()
decoder.eval()
evaluate_randomly(encoder, decoder, input_lang, output_lang, pairs)

Reading lines...
Read 135842 sentence pairs
Trimmed to 11445 sentence pairs
Counting words...
Counted words:
fra 4601
eng 2991
0m 3s (- 0m 50s) (5 6%) 3.4215
0m 6s (- 0m 44s) (10 12%) 2.3431
0m 9s (- 0m 41s) (15 18%) 1.9940
0m 12s (- 0m 37s) (20 25%) 1.7623
0m 15s (- 0m 34s) (25 31%) 1.6099
0m 18s (- 0m 31s) (30 37%) 1.4860
0m 21s (- 0m 28s) (35 43%) 1.3800
0m 25s (- 0m 25s) (40 50%) 1.2873
0m 28s (- 0m 21s) (45 56%) 1.2067
0m 31s (- 0m 18s) (50 62%) 1.1354
0m 34s (- 0m 15s) (55 68%) 1.0702
0m 37s (- 0m 12s) (60 75%) 1.0101
0m 41s (- 0m 9s) (65 81%) 0.9542
0m 44s (- 0m 6s) (70 87%) 0.9014
0m 47s (- 0m 3s) (75 93%) 0.8523
0m 50s (- 0m 0s) (80 100%) 0.8066
> on retourne a la case depart
= we re going back to square one
< we re going to have a new good next year

> tu as probablement soif
= you re probably thirsty
< you re the one who planted in the computer <EOS>

> il est professeur de biologie a harvard
= he s a professor of biology at harvard
< he is a man of a child of problems <EOS>

<Figure size 640x480 with 0 Axes>